# user

In [ ]:
VERSION = "1.1.2_sextractor"

# Sextractor test
# 1.1.0: "4_000",
# 1.1.1: "40_000",
# 1.1.2: "4_000 + min_sersic=0.4",
# hypercube + shear round 2 + 10 gal/patch + shear |0.06| + ellips gaussian + pixels_out

# lib

In [ ]:
from pathlib import Path

from jax import numpy
from polars import DataFrame, read_csv
from tqdm import tqdm
from utils import best_result, case_result, ellips_quad, extra_imgs, histos, plot_sersic, vignettes

DATASET_DIR = Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid") / VERSION
VERSION_DIR = Path("/net/GECO/pisco/PIX2SHEAR/RESULTS_SEXTRACTOR") / VERSION
CATALOG_DIR = VERSION_DIR / "CATALOG"
IMAGES_DIR = VERSION_DIR / "IMAGES"
PLOTS_DIR = VERSION_DIR / "PLOTS"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# images

In [ ]:
CHOICE = "simu_00002"

extra_imgs(
    [
        DATASET_DIR / CHOICE,
        IMAGES_DIR / f"fitting_{CHOICE}_1",
        IMAGES_DIR / f"background_{CHOICE}",
        IMAGES_DIR / f"residual_{CHOICE}_1",
        IMAGES_DIR / f"segmentation_{CHOICE}",
    ],
    ["image", "fitting", "background", "residual", "segmentation"],
    PLOTS_DIR,
)

# vignettes

In [ ]:
CHOICE = "simu_00002"

vignettes(
    PLOTS_DIR,
    csv_file=CATALOG_DIR / f"catalog_{CHOICE}.csv",
    vignettes_file=IMAGES_DIR / f"vignettes_{CHOICE}.fits",
)

# inference

In [ ]:
RESULTS_FILES = sorted(list((CATALOG_DIR).glob("catalog_simu_*.csv")))
NB_TOTAL = len(RESULTS_FILES)

found, avg_e1, avg_e2, avg_e1_quad, avg_e2_quad = [[] for _ in range(5)]
for sample in tqdm(RESULTS_FILES, total=NB_TOTAL):
    df = read_csv(sample)
    found.append(df.shape[0])

    avg_e1.append(df["e1"].to_numpy().mean())
    avg_e2.append(df["e2"].to_numpy().mean())

    e1_quad, e2_quad = ellips_quad(
        df["ellipse_cxx"].to_numpy(), df["ellipse_cyy"].to_numpy(), df["ellipse_cxy"].to_numpy()
    )
    avg_e1_quad.append(e1_quad.mean())
    avg_e2_quad.append(e2_quad.mean())

# save res

In [ ]:
DataFrame(
    {
        "avg_e1": avg_e1,
        "avg_e2": avg_e2,
        "avg_e1_quad": avg_e1_quad,
        "avg_e2_quad": avg_e2_quad,
    },
    schema=[
        "avg_e1",
        "avg_e2",
        "avg_e1_quad",
        "avg_e2_quad",
    ],
).write_csv(VERSION_DIR / "pred_avg_ellips.csv")

del df, e1_quad, e2_quad, avg_e1, avg_e2, avg_e1_quad, avg_e2_quad

# postprocess

In [ ]:
true_images_params = read_csv(DATASET_DIR / "true_images_params.csv", skip_rows=0)
true_avg_ellps = true_images_params["avg_e1", "avg_e2"].to_numpy()
true_avg_shears = true_images_params["avg_g1", "avg_g2"].to_numpy()
pred_avg_ellips = read_csv(VERSION_DIR / "pred_avg_ellips.csv", skip_rows=0).to_numpy()
print(pred_avg_ellips.shape, true_avg_ellps.shape, true_avg_shears.shape)

for i in range(2):
    pred_avg_ellips = pred_avg_ellips[numpy.abs(true_avg_shears[:, i]) <= 0.05]
    true_avg_ellps = true_avg_ellps[numpy.abs(true_avg_shears[:, i]) <= 0.05]
    true_avg_shears = true_avg_shears[numpy.abs(true_avg_shears[:, i]) <= 0.05]
print("restricted range", pred_avg_ellips.shape, true_avg_ellps.shape, true_avg_shears.shape)

del true_images_params

# histo

In [ ]:
bins = 10
histos(preds_ellps=pred_avg_ellips, data_gal=found, directory=PLOTS_DIR, bins=bins)

# best results

In [ ]:
best_result(true_avg_ellps, true_avg_shears, PLOTS_DIR / "best_results.pdf")

# results e

In [ ]:
case = "comp_ell_normal"
case_result(
    pred_avg_ellips[:, 0:2], true_avg_ellps, PLOTS_DIR / f"results_{case}.pdf", symbol="epsilon"
)

case = "comp_ell_quad"
case_result(
    pred_avg_ellips[:, 2:4], true_avg_ellps, PLOTS_DIR / f"results_{case}.pdf", symbol="epsilon"
)

# results e/g

In [ ]:
case = "normal"
case_result(pred_avg_ellips[:, 0:2], true_avg_shears, PLOTS_DIR / f"results_{case}.pdf")

case = "quad"
case_result(pred_avg_ellips[:, 2:4], true_avg_shears, PLOTS_DIR / f"results_{case}.pdf")

# sersic

In [ ]:
X_min, X_max = -5, 5
plot_sersic(X_min, X_max, PLOTS_DIR)
del X_min, X_max

# psf

In [ ]:
# from utils import check_psf
# check_psf(
#     Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid") / "PSF" / "airy.psf.fits", PLOTS_DIR
# )